In [1]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly


In [2]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, re, secrets as pysecrets, smtplib, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# ============================================================
# 🚀 THEME (UNCHANGED — do not modify)
# ============================================================
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8", "success": "#34d399", "danger": "#f87171"
}

# ============================================================
# 🔐 SECRETS (never hard-coded — pulled from environment / Colab Secrets)
# ============================================================
JWT_SECRET = os.getenv("JWT_SECRET", "dev-only-fallback-secret-change-me")
SENDER_EMAIL = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
OTP_EXPIRY_MINUTES = 5

# 🚀 Admin credentials — defined directly in code, NOT a signup account (Milestone Step 11)
ADMIN_USERNAME = "xxxx"
ADMIN_PASSWORD = "xxxx"

SECURITY_QUESTIONS = [
    "What is your pet name?",
    "What is your mother's maiden name?",
    "What is your favourite city?",
    "What was the name of your first school?",
]

# ============================================================
# 🚀 Neo-Brutalist CSS (UNCHANGED — do not modify)
# ============================================================
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}

    /* 🚀 DO NOT HIDE HEADER! Only hide footer and bottom decoration so toggle arrow stays visible! */
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    /* 🚀 Highlight Streamlit's Native Sidebar Re-Open Button (Top-Left) as a Yellow Badge! */
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important; color: {COLORS['text_heading']} !important; stroke: {COLORS['text_heading']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    /* Inputs */
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}

    /* Buttons */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

# ============================================================
# 💾 DATABASE
# ============================================================
def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE,
        email TEXT UNIQUE,
        password_hash TEXT,
        security_question TEXT,
        security_answer_hash TEXT,
        created_at TEXT)""")
    conn.execute("""CREATE TABLE IF NOT EXISTS password_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT,
        password_hash TEXT,
        set_at TEXT)""")

def _now(): return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

def record_password(username, password_hash):
    with get_db() as c:
        c.execute("INSERT INTO password_history (username, password_hash, set_at) VALUES (?, ?, ?)",
                  (username, password_hash, _now()))

def is_password_reused(username, new_password):
    """Milestone requirement: once a password is reset, the old password must not be accepted again."""
    with get_db() as c:
        rows = c.execute("SELECT password_hash FROM password_history WHERE username=?", (username,)).fetchall()
    for (h,) in rows:
        if check_txt(new_password, h):
            return True
    return False

def update_password(username, new_password):
    new_hash = hash_txt(new_password)
    with get_db() as c:
        c.execute("UPDATE users SET password_hash=? WHERE username=?", (new_hash, username))
    record_password(username, new_hash)

def find_user_by_login(identifier):
    """Look up a user by username OR email (Login accepts either)."""
    with get_db() as c:
        return c.execute("SELECT username, email, password_hash FROM users WHERE username=? OR email=?",
                          (identifier, identifier)).fetchone()

def find_user_by_username(username):
    with get_db() as c:
        return c.execute("SELECT username, email, security_question, security_answer_hash FROM users WHERE username=?",
                          (username,)).fetchone()

# ============================================================
# ✅ VALIDATION HELPERS (Milestone Step 8)
# ============================================================
EMAIL_RE = re.compile(r"^[A-Za-z]{2,}[A-Za-z0-9._%+-]*@[A-Za-z]{2,}[A-Za-z0-9.-]*\.[A-Za-z]{2,}$")

def is_valid_email(email):
    return bool(EMAIL_RE.match(email or ""))

def password_rule_errors(pw):
    errs = []
    if len(pw or "") < 8: errs.append("at least 8 characters")
    if not re.search(r"[A-Z]", pw or ""): errs.append("an uppercase letter")
    if not re.search(r"[a-z]", pw or ""): errs.append("a lowercase letter")
    if not re.search(r"[0-9]", pw or ""): errs.append("a number")
    if not re.search(r"[^A-Za-z0-9]", pw or ""): errs.append("a special symbol")
    return errs

# ============================================================
# ✉️ OTP EMAIL HELPERS (Milestone Step 5/9)
# ============================================================
def generate_otp(): return f"{pysecrets.randbelow(900000) + 100000}"

def make_otp_token(username, email, otp):
    payload = {
        "sub": username, "email": email, "otp_hash": hash_txt(otp),
        "type": "password_reset_otp",
        "iat": datetime.datetime.utcnow(),
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, username):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != username or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]):
            return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def send_otp_email(to_email, otp):
    if not SENDER_EMAIL or not EMAIL_PASSWORD:
        return False, "❌ Gmail sender not configured! Add `EMAIL_ADDRESS` and `EMAIL_PASSWORD` to Colab Secrets."
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Portal - Verification Code"
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()
    msg['Reply-To'] = SENDER_EMAIL

    text_body = f"Your verification code for Infosys Portal is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes.\nIf you did not request this code, please ignore this email."
    html_body = f"""
    <html><body style="font-family:Arial,sans-serif;background:#f9fcfc;padding:20px;">
      <div style="max-width:500px;margin:0 auto;background:#fff;border:2px solid #272343;border-radius:12px;padding:30px;text-align:center;">
        <div style="color:#272343;font-size:20px;font-weight:bold;margin-bottom:15px;">Infosys Portal Verification</div>
        <div style="color:#4a5568;font-size:15px;margin-bottom:20px;">We received a request to reset the password for <b>{to_email}</b>.</div>
        <div style="background:#ffd803;color:#272343;font-size:28px;font-weight:bold;letter-spacing:5px;padding:15px 20px;border:2px solid #272343;border-radius:8px;display:inline-block;margin:10px 0;">{otp}</div>
        <div style="color:#4a5568;font-size:15px;margin-top:20px;">This code expires in <b>{OTP_EXPIRY_MINUTES} minutes</b>.</div>
      </div>
    </body></html>
    """
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(SENDER_EMAIL, EMAIL_PASSWORD)
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e:
        return False, f"SMTP Error: {str(e)}"

# ============================================================
# 🔑 JWT SESSION
# ============================================================
def make_jwt(identifier, is_admin=False):
    return jwt.encode({"sub": identifier, "is_admin": is_admin,
                        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)},
                       JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception: return None

# ============================================================
# 🧠 SESSION STATE
# ============================================================
DEFAULT_STATE = {
    "token": None, "page": "Login",
    "forgot_username": None, "forgot_stage": None,
    "forgot_email": None, "forgot_otp_token": None, "forgot_sq": None,
}
for k, v in DEFAULT_STATE.items():
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def reset_forgot_state():
    st.session_state.forgot_username = None
    st.session_state.forgot_stage = None
    st.session_state.forgot_email = None
    st.session_state.forgot_otp_token = None
    st.session_state.forgot_sq = None

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Franchise Analytics and Management Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        # ------------------------------------------------------
        # LOGIN (unchanged UI — now also accepts admin credentials)
        # ------------------------------------------------------
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            identifier = st.text_input("Username or Email address", placeholder="you@infosys.com or username").strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                if not identifier or not pwd:
                    st.error("⚠️ Please fill in both fields.")
                elif identifier == ADMIN_USERNAME and pwd == ADMIN_PASSWORD:
                    st.session_state.token = make_jwt(ADMIN_USERNAME, is_admin=True)
                    navigate("Dashboard")
                else:
                    row = find_user_by_login(identifier.lower())
                    if row and check_txt(pwd, row[2]):
                        st.session_state.token = make_jwt(row[0], is_admin=False)
                        navigate("Dashboard")
                    else:
                        # Generic error — never reveal which field was wrong
                        st.error("❌ Invalid username/email or password.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        # ------------------------------------------------------
        # SIGNUP (unchanged UI, extra validation added)
        # ------------------------------------------------------
        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Username", placeholder="jane_doe").strip()
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", SECURITY_QUESTIONS)
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                pw_errs = password_rule_errors(pwd)
                if not uname or not email or not pwd or not confirm_pwd or not sa:
                    st.error("⚠️ Please fill all fields.")
                elif uname == ADMIN_USERNAME:
                    st.error("❌ That username is reserved.")
                elif not is_valid_email(email):
                    st.error("❌ Please enter a valid email address (e.g. ab@cd.ef).")
                elif pw_errs:
                    st.error("❌ Password must contain " + ", ".join(pw_errs) + ".")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        pwd_hash = hash_txt(pwd)
                        with get_db() as c:
                            c.execute("INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, created_at) VALUES (?, ?, ?, ?, ?, ?)",
                                      (uname, email, pwd_hash, sq, hash_txt(sa.lower().strip()), _now()))
                        record_password(uname, pwd_hash)
                        st.session_state.token = make_jwt(uname, is_admin=False)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Username or email already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        # ------------------------------------------------------
        # FORGOT PASSWORD — Security Question route + OTP route (merged)
        # ------------------------------------------------------
        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            stage = st.session_state.forgot_stage

            # STEP A — ask for username, choose a recovery route
            if stage is None:
                uname_in = st.text_input("Username", placeholder="Enter your username").strip()
                st.markdown("<br>", unsafe_allow_html=True)
                col_sq, col_otp = st.columns(2)

                if col_sq.button("Via Security Question", use_container_width=True):
                    row = find_user_by_username(uname_in)
                    if row:
                        st.session_state.forgot_username = row[0]
                        st.session_state.forgot_sq = row[2]
                        st.session_state.forgot_stage = "sq_answer"
                        st.rerun()
                    else:
                        st.error("❌ Username not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                    row = find_user_by_username(uname_in)
                    if not row:
                        st.error("❌ Username not found.")
                    elif not EMAIL_PASSWORD or not SENDER_EMAIL:
                        st.error("❌ Gmail sender not configured! Add `EMAIL_ADDRESS` / `EMAIL_PASSWORD` to Colab Secrets.")
                    else:
                        registered_email = row[1]
                        otp = generate_otp()
                        with st.spinner(f"Sending 6-digit OTP to registered email (expires in {OTP_EXPIRY_MINUTES} mins)..."):
                            ok, msg = send_otp_email(registered_email, otp)
                        if ok:
                            st.session_state.forgot_username = row[0]
                            st.session_state.forgot_email = registered_email
                            st.session_state.forgot_otp_token = make_otp_token(row[0], registered_email, otp)
                            st.session_state.forgot_stage = "otp_verify"
                            st.success(f"✅ OTP sent to {registered_email[:2]}***@{registered_email.split('@')[-1]}")
                            time.sleep(1); st.rerun()
                        else:
                            st.error(f"❌ {msg}")

            # STEP B1 — Security Question: answer + new password
            elif stage == "sq_answer":
                st.info(f"❓ **Security Question:** {st.session_state.forgot_sq}")
                ans = st.text_input("Your answer").lower().strip()
                npw = st.text_input("New password", type="password", placeholder="Min. 8 characters")
                confirm_npw = st.text_input("Confirm new password", type="password")
                st.markdown("<br>", unsafe_allow_html=True)
                if st.button("Reset Password →", use_container_width=True):
                    pw_errs = password_rule_errors(npw)
                    row = find_user_by_username(st.session_state.forgot_username)
                    if pw_errs:
                        st.error("❌ Password must contain " + ", ".join(pw_errs) + ".")
                    elif npw != confirm_npw:
                        st.error("❌ Passwords do not match.")
                    elif not (row and check_txt(ans, row[3])):
                        st.error("❌ Incorrect security answer.")
                    elif is_password_reused(st.session_state.forgot_username, npw):
                        st.error("❌ You cannot reuse a previous password. Please choose a new one.")
                    else:
                        update_password(st.session_state.forgot_username, npw)
                        st.success("✅ Password updated successfully!")
                        time.sleep(1); reset_forgot_state(); navigate("Login")

            # STEP B2 — OTP: enter code
            elif stage == "otp_verify":
                st.info(f"📧 Code sent to **{st.session_state.forgot_email}** (valid for {OTP_EXPIRY_MINUTES} mins).")
                otp_input = st.text_input("6-Digit Verification Code", max_chars=6, placeholder="e.g. 849201")
                st.markdown("<br>", unsafe_allow_html=True)
                c1, c2 = st.columns([1.2, 1])
                if c1.button("Verify Code →", use_container_width=True):
                    if not otp_input or len(otp_input) != 6:
                        st.error("⚠️ Please enter the valid 6-digit code.")
                    else:
                        ok, msg = verify_otp_token(st.session_state.forgot_otp_token, otp_input, st.session_state.forgot_username)
                        if ok:
                            st.session_state.forgot_stage = "otp_reset"
                            st.success("✅ Code verified successfully!"); time.sleep(1); st.rerun()
                        else:
                            st.error(f"❌ {msg}")
                if c2.button("← Back", use_container_width=True):
                    reset_forgot_state(); st.rerun()

            # STEP B3 — OTP: new password
            elif stage == "otp_reset":
                npw = st.text_input("New password", type="password", placeholder="Min. 8 characters")
                confirm_npw = st.text_input("Confirm new password", type="password")
                st.markdown("<br>", unsafe_allow_html=True)
                if st.button("Update Password →", use_container_width=True):
                    pw_errs = password_rule_errors(npw)
                    if pw_errs:
                        st.error("❌ Password must contain " + ", ".join(pw_errs) + ".")
                    elif npw != confirm_npw:
                        st.error("❌ Passwords do not match.")
                    elif is_password_reused(st.session_state.forgot_username, npw):
                        st.error("❌ You cannot reuse a previous password. Please choose a new one.")
                    else:
                        update_password(st.session_state.forgot_username, npw)
                        st.success("🎉 Password updated successfully! Your account is secure.")
                        time.sleep(1); reset_forgot_state(); navigate("Login")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                reset_forgot_state(); navigate("Login")

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    is_admin = payload.get("is_admin", False)
    identifier = payload["sub"]

    if is_admin:
        uname = "Administrator"
    else:
        with get_db() as c:
            row = c.execute("SELECT username FROM users WHERE username=?", (identifier,)).fetchone()
        uname = row[0] if row else identifier

    # 🚀 SIDEBAR MENU
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if is_admin else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Logout"] if is_admin else ["Dashboard", "Analytics", "Reports", "Logout"]
        icons = ["house", "box-arrow-right"] if is_admin else ["house", "graph-up", "file-text", "box-arrow-right"]
        menu = option_menu(None, opts, icons=icons,
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD
    if is_admin:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        with get_db() as c:
            all_users = c.execute("SELECT username, email, created_at FROM users ORDER BY created_at DESC").fetchall()

        c1, c2 = st.columns([1, 1])
        c1.markdown(f"""<div class="pn-card" style="text-align:center;"><div style="font-size:26px;font-weight:700;">{len(all_users)}</div><div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">Registered Users</div></div>""", unsafe_allow_html=True)
        c2.markdown(f"""<div class="pn-card" style="text-align:center;"><div style="font-size:26px;font-weight:700;">🛡️</div><div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">Admin Session Active</div></div>""", unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown("#### Registered Users")
        search = st.text_input("🔎 Search by username or email", placeholder="Type to filter...")

        filtered = [u for u in all_users if not search or search.lower() in u[0].lower() or search.lower() in u[1].lower()]

        if not filtered:
            st.info("No matching users found.")
        else:
            for username, email, created_at in filtered:
                with st.container(border=True):
                    col_a, col_b, col_c = st.columns([2, 2, 1])
                    col_a.markdown(f"**👤 {username}**")
                    col_b.markdown(f"{email}  \n<span style='color:{COLORS['text_muted']};font-size:12px;'>Joined {created_at}</span>", unsafe_allow_html=True)
                    if col_c.button("🗑️ Delete", key=f"del_{username}", use_container_width=True):
                        with get_db() as c:
                            c.execute("DELETE FROM users WHERE username=?", (username,))
                            c.execute("DELETE FROM password_history WHERE username=?", (username,))
                        st.success(f"Deleted user '{username}'.")
                        time.sleep(1); st.rerun()

    # 🚀 REGULAR USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)


Overwriting app.py


In [ ]:
import os, time, subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Safely retrieve all secrets from Colab Secrets (never hard-code these!)
try:
    os.environ['JWT_SECRET'] = userdata.get('JWT_SECRET')
    os.environ['EMAIL_ADDRESS'] = userdata.get('EMAIL_ADDRESS')
    os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
except Exception as e:
    print("⚠️ Secret Error: Make sure JWT_SECRET, EMAIL_ADDRESS, EMAIL_PASSWORD and NGROK_AUTHTOKEN")
    print("   are all added under the Colab Secrets (key icon) tab, with notebook access enabled.")

# 2. Kill any previous Streamlit server (ngrok tunnel is reused if still alive)
ngrok.kill()
get_ipython().system('pkill -f streamlit')
time.sleep(2)

# 3. Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)

# 4. Open the public ngrok tunnel
try:
    public_url = ngrok.connect(8501).public_url
    print("=" * 65)
    print(f"👉 Infosys Portal Live URL: {public_url}")
    print("=" * 65)
    print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down server...")
    ngrok.kill()
    process.terminate()
    get_ipython().system('pkill -f streamlit')
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")


👉 Infosys Portal Live URL: https://nest-unlatch-pope.ngrok-free.dev
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.
